# Depressive and Suicidal text classification

**Datasets used:**
1. [Classified Twitter posts](https://www.kaggle.com/datasets/munkialbright/classified-tweets)
2. [Derived dataset from Murarka et al., Nikhileswar Komati and Suchintika Sarkar](https://www.kaggle.com/datasets/priyangshumukherjee/mental-health-text-classification-dataset)

## Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

## Download the dataset

In [2]:
import kagglehub

In [3]:
path1 = kagglehub.dataset_download("munkialbright/classified-tweets")
print("Path to dataset 1:", path1)

Using Colab cache for faster access to the 'classified-tweets' dataset.
Path to dataset 1: /kaggle/input/classified-tweets


In [4]:
path2 = kagglehub.dataset_download("priyangshumukherjee/mental-health-text-classification-dataset")
print("Path to dataset 2:", path2)

Using Colab cache for faster access to the 'mental-health-text-classification-dataset' dataset.
Path to dataset 2: /kaggle/input/mental-health-text-classification-dataset


## Set up dataframes

In [5]:
# We do multi-label (binary label) classification
# with possible missing values in the labels
# 0 = not present, 1 = present, -1 = unknown/missing
# normal text have all labels = 0

df_all = pd.DataFrame(columns=["text", "isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"])

In [6]:
df_all1 = pd.read_csv(path1 + "/classified_tweets.csv")
# print columns
print("Columns in dataset 1:", df_all1.columns)

# cols = 
# 'text' -> text of the tweet 
# 'suspicious' -> 0 or 1 if the tweet is suspicious (contains harmful content)
# 'cyberbullying' -> 0 if nothing, 1 if racist, 2 if sexist
# 'hate' -> 0 or 1 if the tweet contains hate speech
# 'suicidal' -> 0 or 1 if the tweet contains suicidal content

# this dataset has no depressive / anxiety labels, so we will fill them with -1 (unknown/missing)
df_all1["isDepressive"] = -1
df_all1["isAnxiety"] = -1
# rename columns to match our format
df_all1.rename(columns={
    "suspicious": "isSuspicious", 
    "hate": "isHate", 
    "suicidal": "isSuicidal"
}, inplace=True)

# convert cyberbullying to isRacist and isSexist
df_all1["isRacist"] = df_all1["cyberbullying"].apply(lambda x: 1 if x == 1 else 0)
df_all1["isSexist"] = df_all1["cyberbullying"].apply(lambda x: 1 if x == 2 else 0)
# drop the original cyberbullying column
df_all1.drop(columns=["cyberbullying"], inplace=True)

display(df_all1.head())

df_all = pd.concat([df_all, df_all1], ignore_index=True)

# debug, print unique values of row isSexist
print("Unique values in isSexist:", df_all["isSexist"].unique())
# count how many rows have isSexist = 1
print("Number of rows with isSexist = 1:", (df_all["isSexist"] == 1).sum())

Columns in dataset 1: Index(['text', 'suspicious', 'cyberbullying', 'hate', 'suicidal'], dtype='object')


,text,isSuspicious,isHate,isSuicidal,isDepressive,isAnxiety,isRacist,isSexist
0,Uhmm like 6th grade on a corner of a street....,0,0,0,-1,-1,0,0
1,a) JTP is a douchebag b) Stewart kicks ass!,1,0,0,-1,-1,0,0
2,ditto bitch!,1,0,0,-1,-1,0,0
3,damn I have to drive my dad to the airport tha...,0,0,0,-1,-1,0,0
4,:],0,0,0,-1,-1,0,0


Unique values in isSexist: [0 1]
Number of rows with isSexist = 1: 1733


In [7]:
df_all2 = pd.read_csv(path2 + "/mental_heath_unbanlanced.csv")

# print columns
print("Columns in dataset 2:", df_all2.columns)

# cols =
# 'text' -> text of the post
# 'status' -> the text Normal, Depression, Suicidal, Anxiety

def getRow(
        default_num : int,
        change_key : str = None,
        change_value : int = None
    ):
    res = {
        "isSuspicious": default_num, 
        "isRacist": default_num, 
        "isSexist": default_num, 
        "isHate": default_num, 
        "isDepressive": default_num, 
        "isSuicidal": default_num, 
        "isAnxiety": default_num,
    }
    if change_key is not None and change_value is not None:
        res[change_key] = change_value
    return res

mapping_cols = {
    "Normal"     : getRow(0),
    "Depression" : getRow(-1, "isDepressive", 1),
    "Suicidal"   : getRow(-1, "isSuicidal", 1),
    "Anxiety"    : getRow(-1, "isAnxiety", 1),
}

# apply the mapping to the status column and create new columns
for status, mapping in mapping_cols.items():
    df_all2.loc[df_all2["status"] == status, ["isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"]] = mapping.values()

# drop the original status column
df_all2.drop(columns=["status", "Unique_ID"], inplace=True)

display(df_all2.head())

df_all = pd.concat([df_all, df_all2], ignore_index=True)

Columns in dataset 2: Index(['Unique_ID', 'text', 'status'], dtype='object')


,text,isSuspicious,isRacist,isSexist,isHate,isDepressive,isSuicidal,isAnxiety
0,oh my gosh,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0
1,"trouble sleeping, confused mind, restless hear...",-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0
2,"All wrong, back off dear, forward doubt. Stay ...",-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0
3,I've shifted my focus to something else but I'...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0
4,"I'm restless and restless, it's been a month n...",-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0


In [8]:
del df_all1
del df_all2

In [9]:
# turn any value > 1 in whatever col into 1
for col in ["isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"]:
    df_all[col] = df_all[col].apply(lambda x: 1 if x > 1 else x)

## Print stat

In [10]:
num_samples = len(df_all)
print("Total number of samples:", num_samples)

Total number of samples: 69546


In [11]:
count_df = df_all[["isSuspicious", "isRacist", "isSexist", "isHate", "isDepressive", "isSuicidal", "isAnxiety"]].apply(pd.Series.value_counts).T
print("Label distribution:")
print(count_df)

Label distribution:
               -1.0    0.0    1.0
isSuspicious  31221  31110   7215
isRacist      31221  37380    945
isSexist      31221  36592   1733
isHate        31221  35647   2678
isDepressive  36649  18391  14506
isSuicidal    20009  37277  12260
isAnxiety     45652  18391   5503
